In [1]:
import pandas as pd
from mopac_batch_runner import prepare_batch, run_mopac_batch, export_sdf

In [2]:
df = pd.read_csv('./datasets/12k_pfas_canonical.csv')
df

,compound_id,SMILES,canonical_SMILES
0,100016-68-0,FC(F)(OC(\F)=C(\F)F)C(F)(F)OC(F)(C(=O)OC)C(F)(F)F,COC(=O)C(F)(OC(F)(F)C(F)(F)OC(F)=C(F)F)C(F)(F)F
1,1000-28-8,FC(C(F)(F)OCC(F)(F)C(F)(F)F)C(F)(F)F,FC(C(F)(F)F)C(F)(F)OCC(F)(F)C(F)(F)F
2,1000339-60-5,ClCc1ccc(cc1F)C(F)(F)F,Fc1cc(C(F)(F)F)ccc1CCl
3,1000509-29-4,FC(F)(C(F)(F)CCN)C(F)(F)F,NCCC(F)(F)C(F)(F)C(F)(F)F
4,1000572-70-2,FC(F)(F)c1ccc(cc1CCO)C(F)(F)F,OCCc1cc(C(F)(F)F)ccc1C(F)(F)F
...,...,...,...
12644,NOCAS_894124,O=C(CCSCCC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(...,CC(NC(=O)CCSCCC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)...
12645,NOCAS_894130,FC(F)(CCSCCC(=O)OC(C)C(=O)O)C(F)(F)C(F)(F)C(F)...,CC(OC(=O)CCSCCC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)...
12646,NOCAS_894131,FC(F)(CCS(=O)CCC(=O)O)C(F)(F)C(F)(F)C(F)(F)C(F...,O=C(O)CCS(=O)CCC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F...
12647,NOCAS_896897,FC(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F...,C=CC(=O)OCCNCS(=O)(=O)C(F)(F)C(F)(F)C(F)(F)C(F...


In [3]:
df_test = df.head(500)

In [4]:
work_dir = 'pfas_test'

In [5]:
# Step 1: SMILES → 3D structures → .mop files
prep = prepare_batch(
    df=df_test,
    smiles_col="canonical_SMILES",
    id_col="compound_id",
    keywords="PM7 PRECISE GEO-OK",
    workdir=work_dir,
    auto_charge=True,   # ← detects [NH4+], [O-], etc. automatically
    overwrite=True,
    max_workers=8
)

In [6]:
# verify if anything failed
print(prep[prep['status'] != 'ready'])

     compound_id                                             smiles  status  \
419  106065-05-8  [O+]#[C-]->[Cr](<-[C-]#[O+])(<-[C-]#[O+])(<-[C...  failed   

    mop_file                                      error_message  duration_s  
419      NaN  Open Babel returned rc=0 but structure.mol2 is...       0.039  


In [7]:
# Step 2: run MOPAC on all prepared .mop files
results = run_mopac_batch(
    workdir=work_dir,
    overwrite=True,
    timeout=900,
    max_workers=8,
)

[1065-49-2] MOPAC timed out.


In [8]:
results[results['arc_file'].isna()]#.to_excel('twice_failed_molecules.xlsx')
results[results['arc_file'].isna()]['compound_id']

12    1065-49-2
Name: compound_id, dtype: str

In [10]:
# Step 3: export optimised geometries to SDF
sdf = export_sdf(
    workdir=work_dir,
    overwrite=True,
    max_workers=6,
)

In [11]:
print(sdf[sdf['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, arc_file, sdf_file, status, error_message, duration_s]
Index: []


In [19]:
from pathlib import Path
import shutil

source_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\mopac_jobs")
target_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\sdf_all")

target_dir.mkdir(exist_ok=True)

for sdf_file in source_dir.rglob("*.sdf"):
    shutil.copy2(sdf_file, target_dir / sdf_file.name)